# ==========================================
# GOLD MART 1 - AIRPORT OPERATIONS
# ==========================================

In [1]:
from pyspark.sql.functions import *

fact_flights = spark.table("fact_flights")

dim_airport = spark.table("dim_airport")

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 3, Finished, Available, Finished, False)

In [2]:
fact_flights.printSchema()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 4, Finished, Available, Finished, False)

root
 |-- flight_key: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- airline_key: integer (nullable = true)
 |-- origin_airport_key: integer (nullable = true)
 |-- destination_airport_key: integer (nullable = true)
 |-- route_key: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)



In [3]:
dim_airport.printSchema()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 5, Finished, Available, Finished, False)

root
 |-- airport_key: integer (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- airport_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



# Airport Fact View

In [4]:
airport_perf = (
    fact_flights.alias("f")
    .join(
        dim_airport.alias("a"),
        col("f.origin_airport_key") ==
        col("a.airport_key"),
        "left"
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 6, Finished, Available, Finished, False)

In [5]:
print(airport_perf.columns)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 7, Finished, Available, Finished, False)

['flight_key', 'date_key', 'airline_key', 'origin_airport_key', 'destination_airport_key', 'route_key', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'AIR_TIME', 'airport_key', 'iata_code', 'airport_name', 'city', 'country', 'latitude', 'longitude']


In [6]:
airport_perf.filter(
    col("airport_name").isNull()
).count()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 8, Finished, Available, Finished, False)

149

In [7]:
airport_perf = (
    airport_perf
    .withColumn(
        "delay_flag",
        when(col("DEP_DELAY") > 15, 1)
        .otherwise(0)
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 9, Finished, Available, Finished, False)

In [8]:
airport_perf.groupBy(
    "delay_flag"
).count().show()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 10, Finished, Available, Finished, False)

+----------+------+
|delay_flag| count|
+----------+------+
|         1|118522|
|         0|428749|
+----------+------+



# gold_airport_operations

In [9]:
gold_airport_operations = (
    airport_perf
    .groupBy(
        "origin_airport_key",
        "airport_name",
        "city",
        "country"
    )
    .agg(
        count("*").alias("total_flights"),

        sum("delay_flag").alias("delayed_flights"),

        round(
            avg("DEP_DELAY"),
            2
        ).alias("avg_dep_delay"),

        round(
            avg("ARR_DELAY"),
            2
        ).alias("avg_arr_delay"),

        sum("CANCELLED").alias("cancelled_flights"),

        sum("DIVERTED").alias("diverted_flights"),

        round(
            sum("DEP_DELAY"),
            2
        ).alias("total_delay_minutes")
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 11, Finished, Available, Finished, False)

In [10]:
gold_airport_operations.show(10)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 12, Finished, Available, Finished, False)

+------------------+--------------------+-------------+-------------+-------------+---------------+-------------+-------------+-----------------+----------------+-------------------+
|origin_airport_key|        airport_name|         city|      country|total_flights|delayed_flights|avg_dep_delay|avg_arr_delay|cancelled_flights|diverted_flights|total_delay_minutes|
+------------------+--------------------+-------------+-------------+-------------+---------------+-------------+-------------+-----------------+----------------+-------------------+
|              1584|Fort Dodge Region...|   Fort Dodge|United States|           52|             13|        21.93|        12.18|              7.0|             1.0|              987.0|
|              2656|Ketchikan Interna...|    Ketchikan|United States|          182|             37|         5.35|         0.04|             12.0|             4.0|              909.0|
|              4521|San Francisco Int...|San Francisco|United States|        10133|  

In [11]:
gold_airport_operations = (
    gold_airport_operations
    .withColumn(
        "delay_rate_pct",
        round(
            (col("delayed_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate_pct",
        round(
            (col("cancelled_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "diversion_rate_pct",
        round(
            (col("diverted_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "on_time_flights",
        col("total_flights") - col("delayed_flights")
    )
    .withColumn(
        "on_time_rate_pct",
        round(
            (
                (col("total_flights") - col("delayed_flights"))
                / col("total_flights")
            ) * 100,
            2
        )
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 13, Finished, Available, Finished, False)

In [12]:
gold_airport_operations.select(
    "airport_name",
    "total_flights",
    "delay_rate_pct",
    "cancellation_rate_pct",
    "on_time_rate_pct"
).show(10, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 14, Finished, Available, Finished, False)

+------------------------------------------+-------------+--------------+---------------------+----------------+
|airport_name                              |total_flights|delay_rate_pct|cancellation_rate_pct|on_time_rate_pct|
+------------------------------------------+-------------+--------------+---------------------+----------------+
|Fort Dodge Regional Airport               |52           |25.0          |13.46                |75.0            |
|Ketchikan International Airport           |182          |20.33         |6.59                 |79.67           |
|San Francisco International Airport       |10133        |19.89         |5.66                 |80.11           |
|Flagstaff Pulliam Airport                 |110          |15.45         |0.0                  |84.55           |
|Eastern Sierra Regional Airport           |54           |24.07         |3.7                  |75.93           |
|Monterey Peninsula Airport                |309          |13.59         |0.32                 |8

In [13]:
gold_airport_operations = (
    gold_airport_operations
    .withColumn(
        "operational_disruption_score",
        round(
            (col("delay_rate_pct") * 0.5)
            +
            (col("cancellation_rate_pct") * 0.3)
            +
            (col("diversion_rate_pct") * 0.2),
            2
        )
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 15, Finished, Available, Finished, False)

In [14]:
gold_airport_operations.select(
    "airport_name",
    "delay_rate_pct",
    "cancellation_rate_pct",
    "diversion_rate_pct",
    "operational_disruption_score"
).show(10, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 16, Finished, Available, Finished, False)

+------------------------------------------+--------------+---------------------+------------------+----------------------------+
|airport_name                              |delay_rate_pct|cancellation_rate_pct|diversion_rate_pct|operational_disruption_score|
+------------------------------------------+--------------+---------------------+------------------+----------------------------+
|Fort Dodge Regional Airport               |25.0          |13.46                |1.92              |16.92                       |
|Ketchikan International Airport           |20.33         |6.59                 |2.2               |12.58                       |
|San Francisco International Airport       |19.89         |5.66                 |0.25              |11.69                       |
|Flagstaff Pulliam Airport                 |15.45         |0.0                  |0.0               |7.73                        |
|Eastern Sierra Regional Airport           |24.07         |3.7                  |1.85     

In [15]:
from pyspark.sql.functions import when

gold_airport_operations = (
    gold_airport_operations
    .withColumn(
        "airport_tier",
        when(col("total_flights") >= 5000, "High Volume")
        .when(col("total_flights") >= 1000, "Medium Volume")
        .otherwise("Low Volume")
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 17, Finished, Available, Finished, False)

In [16]:
gold_airport_operations.groupBy(
    "airport_tier"
).count().show()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 18, Finished, Available, Finished, False)

+-------------+-----+
| airport_tier|count|
+-------------+-----+
|  High Volume|   31|
|   Low Volume|  248|
|Medium Volume|   55|
+-------------+-----+



In [17]:
gold_airport_operations.groupBy(
    "airport_tier"
).count().show()

print(
    "Total Airports:",
    gold_airport_operations.count()
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 19, Finished, Available, Finished, False)

+-------------+-----+
| airport_tier|count|
+-------------+-----+
|  High Volume|   31|
|   Low Volume|  248|
|Medium Volume|   55|
+-------------+-----+

Total Airports: 334


In [18]:
gold_airport_operations.agg(
    count("*").alias("total_airports"),
    sum("total_flights").alias("network_flights"),
    avg("delay_rate_pct").alias("avg_network_delay_rate")
).show()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 20, Finished, Available, Finished, False)

+--------------+---------------+----------------------+
|total_airports|network_flights|avg_network_delay_rate|
+--------------+---------------+----------------------+
|           334|         547271|    19.776437125748497|
+--------------+---------------+----------------------+



In [19]:
gold_airport_operations.write \
.mode("overwrite") \
.saveAsTable("gold_airport_operations")

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 21, Finished, Available, Finished, False)

In [20]:
spark.table("gold_airport_operations").count()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 22, Finished, Available, Finished, False)

334

# ==========================================
# GOLD MART 2 - AIRLINE OPERATIONS
# ==========================================

In [21]:
dim_airline = spark.table("dim_airline")

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 23, Finished, Available, Finished, False)

In [22]:
airline_perf = (
    fact_flights.alias("f")
    .join(
        dim_airline.alias("a"),
        col("f.airline_key") ==
        col("a.airline_key"),
        "left"
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 24, Finished, Available, Finished, False)

In [23]:
print(airline_perf.columns)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 25, Finished, Available, Finished, False)

['flight_key', 'date_key', 'airline_key', 'origin_airport_key', 'destination_airport_key', 'route_key', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'AIR_TIME', 'airline_key', 'iata_code', 'airline_name', 'country']


In [24]:
airline_perf = (
    airline_perf
    .withColumn(
        "delay_flag",
        when(col("DEP_DELAY") > 15, 1)
        .otherwise(0)
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 26, Finished, Available, Finished, False)

In [25]:
print(airline_perf.columns)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 27, Finished, Available, Finished, False)

['flight_key', 'date_key', 'airline_key', 'origin_airport_key', 'destination_airport_key', 'route_key', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'AIR_TIME', 'airline_key', 'iata_code', 'airline_name', 'country', 'delay_flag']


In [26]:
airline_perf = airline_perf.select(
    col("f.flight_key"),
    col("f.date_key"),
    col("f.airline_key"),
    col("f.origin_airport_key"),
    col("f.destination_airport_key"),
    col("f.route_key"),
    col("f.DEP_DELAY"),
    col("f.ARR_DELAY"),
    col("f.CANCELLED"),
    col("f.DIVERTED"),
    col("f.DISTANCE"),
    col("f.AIR_TIME"),
    col("a.iata_code"),
    col("a.airline_name"),
    col("a.country"),
    col("delay_flag")
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 28, Finished, Available, Finished, False)

In [27]:
print(airline_perf.columns)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 29, Finished, Available, Finished, False)

['flight_key', 'date_key', 'airline_key', 'origin_airport_key', 'destination_airport_key', 'route_key', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'AIR_TIME', 'iata_code', 'airline_name', 'country', 'delay_flag']


In [28]:
gold_airline_operations = (
    airline_perf
    .groupBy(
        "airline_key",
        "iata_code",
        "airline_name",
        "country"
    )
    .agg(
        count("*").alias("total_flights"),

        sum("delay_flag").alias("delayed_flights"),

        round(
            avg("DEP_DELAY"),
            2
        ).alias("avg_dep_delay"),

        round(
            avg("ARR_DELAY"),
            2
        ).alias("avg_arr_delay"),

        sum("CANCELLED").alias("cancelled_flights"),

        sum("DIVERTED").alias("diverted_flights"),

        round(
            sum("DEP_DELAY"),
            2
        ).alias("total_delay_minutes")
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 30, Finished, Available, Finished, False)

In [29]:
gold_airline_operations.show(10, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 31, Finished, Available, Finished, False)

+-----------+---------+-----------------+-------------+-------------+---------------+-------------+-------------+-----------------+----------------+-------------------+
|airline_key|iata_code|airline_name     |country      |total_flights|delayed_flights|avg_dep_delay|avg_arr_delay|cancelled_flights|diverted_flights|total_delay_minutes|
+-----------+---------+-----------------+-------------+-------------+---------------+-------------+-------------+-----------------+----------------+-------------------+
|1028       |OO       |SkyWest          |United States|56814        |11568          |19.8         |15.2         |2384.0           |224.0           |1078738.0          |
|351        |B6       |JetBlue Airways  |United States|19580        |5213           |19.15        |13.52        |334.0            |74.0            |368880.0           |
|1020       |OH       |Comair           |United States|16526        |3536           |18.77        |14.99        |944.0            |43.0            |293244.

In [30]:
gold_airline_operations = (
    gold_airline_operations
    .withColumn(
        "delay_rate_pct",
        round(
            (col("delayed_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate_pct",
        round(
            (col("cancelled_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "diversion_rate_pct",
        round(
            (col("diverted_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "on_time_flights",
        col("total_flights") - col("delayed_flights")
    )
    .withColumn(
        "on_time_rate_pct",
        round(
            (
                (col("total_flights") - col("delayed_flights"))
                / col("total_flights")
            ) * 100,
            2
        )
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 32, Finished, Available, Finished, False)

In [31]:
gold_airline_operations.select(
    "airline_name",
    "total_flights",
    "delay_rate_pct",
    "cancellation_rate_pct",
    "on_time_rate_pct"
).show(10, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 33, Finished, Available, Finished, False)

+-----------------+-------------+--------------+---------------------+----------------+
|airline_name     |total_flights|delay_rate_pct|cancellation_rate_pct|on_time_rate_pct|
+-----------------+-------------+--------------+---------------------+----------------+
|SkyWest          |56814        |20.36         |4.2                  |79.64           |
|JetBlue Airways  |19580        |26.62         |1.71                 |73.38           |
|Comair           |16526        |21.4          |5.71                 |78.6            |
|American Airlines|77346        |26.1          |1.61                 |73.9            |
|Delta Air Lines  |74384        |17.31         |0.64                 |82.69           |
|Spirit Airlines  |20415        |24.86         |1.48                 |75.14           |
|Midwest Airlines |22914        |13.31         |4.63                 |86.69           |
|United Airlines  |58855        |18.94         |8.01                 |81.06           |
|Pinnacle Airlines|16972        

In [32]:
gold_airline_operations = (
    gold_airline_operations
    .withColumn(
        "operational_disruption_score",
        round(
            (col("delay_rate_pct") * 0.5)
            +
            (col("cancellation_rate_pct") * 0.3)
            +
            (col("diversion_rate_pct") * 0.2),
            2
        )
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 34, Finished, Available, Finished, False)

In [33]:
gold_airline_operations = (
    gold_airline_operations
    .withColumn(
        "airline_tier",
        when(col("total_flights") >= 50000, "Major Carrier")
        .when(col("total_flights") >= 15000, "Regional Carrier")
        .otherwise("Niche Carrier")
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 35, Finished, Available, Finished, False)

In [34]:
gold_airline_operations.select(
    "airline_name",
    "total_flights",
    "airline_tier",
    "operational_disruption_score"
).show(10, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 36, Finished, Available, Finished, False)

+-----------------+-------------+----------------+----------------------------+
|airline_name     |total_flights|airline_tier    |operational_disruption_score|
+-----------------+-------------+----------------+----------------------------+
|SkyWest          |56814        |Major Carrier   |11.52                       |
|JetBlue Airways  |19580        |Regional Carrier|13.9                        |
|Comair           |16526        |Regional Carrier|12.46                       |
|American Airlines|77346        |Major Carrier   |13.59                       |
|Delta Air Lines  |74384        |Major Carrier   |8.9                         |
|Spirit Airlines  |20415        |Regional Carrier|12.9                        |
|Midwest Airlines |22914        |Regional Carrier|8.09                        |
|United Airlines  |58855        |Major Carrier   |11.92                       |
|Pinnacle Airlines|16972        |Regional Carrier|11.34                       |
|Hawaiian Airlines|6576         |Niche C

In [35]:
gold_airline_operations.select(
    "airline_name",
    "iata_code",
    "total_flights"
).orderBy(
    col("total_flights").desc()
).show(20, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 37, Finished, Available, Finished, False)

+-----------------------+---------+-------------+
|airline_name           |iata_code|total_flights|
+-----------------------+---------+-------------+
|Southwest Airlines     |WN       |115389       |
|American Airlines      |AA       |77346        |
|Delta Air Lines        |DL       |74384        |
|United Airlines        |UA       |58855        |
|SkyWest                |OO       |56814        |
|Midwest Airlines       |YX       |22914        |
|American Eagle Airlines|MQ       |20750        |
|Spirit Airlines        |NK       |20415        |
|JetBlue Airways        |B6       |19580        |
|Alaska Airlines        |AS       |17775        |
|Pinnacle Airlines      |9E       |16972        |
|Comair                 |OH       |16526        |
|Frontier Airlines      |F9       |14379        |
|Allegiant Air          |G4       |8596         |
|Hawaiian Airlines      |HA       |6576         |
+-----------------------+---------+-------------+



In [36]:
print(
    "Total Airlines:",
    gold_airline_operations.count()
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 38, Finished, Available, Finished, False)

Total Airlines: 15


In [37]:
gold_airline_operations.groupBy(
    "airline_tier"
).count().show()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 39, Finished, Available, Finished, False)

+----------------+-----+
|    airline_tier|count|
+----------------+-----+
|   Niche Carrier|    3|
|   Major Carrier|    5|
|Regional Carrier|    7|
+----------------+-----+



In [38]:
gold_airline_operations.write \
.mode("overwrite") \
.saveAsTable("gold_airline_operations")

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 40, Finished, Available, Finished, False)

In [39]:
spark.table("gold_airline_operations").count()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 41, Finished, Available, Finished, False)

15

# Route Dimension

In [40]:
dim_route = spark.table("dim_route")

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 42, Finished, Available, Finished, False)

In [41]:
route_perf = (
    fact_flights.alias("f")
    .join(
        dim_route.alias("r"),
        col("f.route_key") ==
        col("r.route_key"),
        "left"
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 43, Finished, Available, Finished, False)

In [42]:
route_perf = (
    route_perf
    .withColumn(
        "delay_flag",
        when(col("DEP_DELAY") > 15, 1)
        .otherwise(0)
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 44, Finished, Available, Finished, False)

In [43]:
print(route_perf.columns)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 45, Finished, Available, Finished, False)

['flight_key', 'date_key', 'airline_key', 'origin_airport_key', 'destination_airport_key', 'route_key', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'AIR_TIME', 'origin_airport', 'destination_airport', 'route_key', 'delay_flag']


In [44]:
route_perf = route_perf.select(
    col("f.flight_key"),
    col("f.date_key"),
    col("f.airline_key"),
    col("f.origin_airport_key"),
    col("f.destination_airport_key"),
    col("f.route_key"),
    col("f.DEP_DELAY"),
    col("f.ARR_DELAY"),
    col("f.CANCELLED"),
    col("f.DIVERTED"),
    col("f.DISTANCE"),
    col("f.AIR_TIME"),
    col("r.origin_airport"),
    col("r.destination_airport"),
    col("delay_flag")
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 46, Finished, Available, Finished, False)

In [45]:
print(route_perf.columns)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 47, Finished, Available, Finished, False)

['flight_key', 'date_key', 'airline_key', 'origin_airport_key', 'destination_airport_key', 'route_key', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'AIR_TIME', 'origin_airport', 'destination_airport', 'delay_flag']


In [46]:
gold_route_operations = (
    route_perf
    .groupBy(
        "route_key",
        "origin_airport",
        "destination_airport"
    )
    .agg(
        count("*").alias("total_flights"),

        sum("delay_flag").alias("delayed_flights"),

        round(
            avg("DEP_DELAY"),
            2
        ).alias("avg_dep_delay"),

        round(
            avg("ARR_DELAY"),
            2
        ).alias("avg_arr_delay"),

        sum("CANCELLED").alias("cancelled_flights"),

        sum("DIVERTED").alias("diverted_flights"),

        round(
            sum("DEP_DELAY"),
            2
        ).alias("total_delay_minutes")
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 48, Finished, Available, Finished, False)

In [47]:
gold_route_operations.show(10, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 49, Finished, Available, Finished, False)

+---------+--------------+-------------------+-------------+---------------+-------------+-------------+-----------------+----------------+-------------------+
|route_key|origin_airport|destination_airport|total_flights|delayed_flights|avg_dep_delay|avg_arr_delay|cancelled_flights|diverted_flights|total_delay_minutes|
+---------+--------------+-------------------+-------------+---------------+-------------+-------------+-----------------+----------------+-------------------+
|3830     |ONT           |OAK                |135          |13             |2.9          |-3.94        |0.0              |0.0             |392.0              |
|632      |BOS           |MCI                |31           |8              |12.29        |-3.03        |0.0              |1.0             |381.0              |
|3004     |LGA           |EYW                |61           |7              |4.84         |-1.28        |3.0              |0.0             |281.0              |
|2971     |LEX           |ORD           

In [48]:
gold_route_operations = (
    gold_route_operations
    .withColumn(
        "delay_rate_pct",
        round(
            (col("delayed_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate_pct",
        round(
            (col("cancelled_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "diversion_rate_pct",
        round(
            (col("diverted_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "on_time_flights",
        col("total_flights") - col("delayed_flights")
    )
    .withColumn(
        "on_time_rate_pct",
        round(
            (
                (col("total_flights") - col("delayed_flights"))
                / col("total_flights")
            ) * 100,
            2
        )
    )
    .withColumn(
        "operational_disruption_score",
        round(
            (col("delay_rate_pct") * 0.5)
            +
            (col("cancellation_rate_pct") * 0.3)
            +
            (col("diversion_rate_pct") * 0.2),
            2
        )
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 50, Finished, Available, Finished, False)

In [49]:
gold_route_operations.select(
    "origin_airport",
    "destination_airport",
    "total_flights",
    "route_tier",
    "operational_disruption_score"
).show(10, False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, 51, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `route_tier` cannot be resolved. Did you mean one of the following? [`f`.`route_key`, `on_time_flights`, `total_flights`, `avg_dep_delay`, `delay_rate_pct`].;
'Project [origin_airport#7014, destination_airport#7015, total_flights#7099L, 'route_tier, operational_disruption_score#7702]
+- Project [route_key#761, origin_airport#7014, destination_airport#7015, total_flights#7099L, delayed_flights#7101L, avg_dep_delay#7103, avg_arr_delay#7105, cancelled_flights#7107, diverted_flights#7109, total_delay_minutes#7111, delay_rate_pct#7632, cancellation_rate_pct#7644, diversion_rate_pct#7657, on_time_flights#7671L, on_time_rate_pct#7686, round((((delay_rate_pct#7632 * 0.5) + (cancellation_rate_pct#7644 * 0.3)) + (diversion_rate_pct#7657 * 0.2)), 2) AS operational_disruption_score#7702]
   +- Project [route_key#761, origin_airport#7014, destination_airport#7015, total_flights#7099L, delayed_flights#710

In [ ]:
gold_route_operations.write \
.mode("overwrite") \
.saveAsTable("gold_route_operations")

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, -1, Cancelled, , Cancelled, True)

In [ ]:
spark.table("gold_route_operations").count()

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, -1, Cancelled, , Cancelled, True)

# gold_network_summary

In [ ]:
gold_network_summary = (
    fact_flights
    .agg(
        count("*").alias("total_flights"),

        sum(
            when(col("DEP_DELAY") > 15, 1)
            .otherwise(0)
        ).alias("delayed_flights"),

        sum("CANCELLED")
        .alias("cancelled_flights"),

        sum("DIVERTED")
        .alias("diverted_flights"),

        round(
            avg("DEP_DELAY"),
            2
        ).alias("avg_dep_delay"),

        round(
            avg("ARR_DELAY"),
            2
        ).alias("avg_arr_delay"),

        round(
            sum("DEP_DELAY"),
            2
        ).alias("total_delay_minutes")
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, -1, Cancelled, , Cancelled, True)

In [ ]:
gold_network_summary.show(truncate=False)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, -1, Cancelled, , Cancelled, True)

In [ ]:
gold_network_summary = (
    gold_network_summary
    .withColumn(
        "delay_rate_pct",
        round(
            (col("delayed_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate_pct",
        round(
            (col("cancelled_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "diversion_rate_pct",
        round(
            (col("diverted_flights") / col("total_flights")) * 100,
            2
        )
    )
    .withColumn(
        "on_time_flights",
        col("total_flights") - col("delayed_flights")
    )
    .withColumn(
        "on_time_rate_pct",
        round(
            (
                (col("total_flights") - col("delayed_flights"))
                / col("total_flights")
            ) * 100,
            2
        )
    )
    .withColumn(
        "network_disruption_score",
        round(
            (col("delay_rate_pct") * 0.5)
            +
            (col("cancellation_rate_pct") * 0.3)
            +
            (col("diversion_rate_pct") * 0.2),
            2
        )
    )
)

StatementMeta(, fbbf915c-d697-4b2b-8d62-152c1c9e1048, -1, Cancelled, , Cancelled, True)

In [110]:
gold_network_summary.show(truncate=False)

StatementMeta(, 1815d0f5-814b-4c4d-8bfc-dc6a5bd3a614, 117, Finished, Available, Finished, False)

+-------------+---------------+-----------------+----------------+-------------+-------------+-------------------+--------------+---------------------+------------------+---------------+----------------+------------------------+
|total_flights|delayed_flights|cancelled_flights|diverted_flights|avg_dep_delay|avg_arr_delay|total_delay_minutes|delay_rate_pct|cancellation_rate_pct|diversion_rate_pct|on_time_flights|on_time_rate_pct|network_disruption_score|
+-------------+---------------+-----------------+----------------+-------------+-------------+-------------------+--------------+---------------------+------------------+---------------+----------------+------------------------+
|547271       |118522         |20389.0          |1512.0          |15.7         |10.35        |8280743.0          |21.66         |3.73                 |0.28              |428749         |78.34           |12.0                    |
+-------------+---------------+-----------------+----------------+-------------+----

In [111]:
gold_network_summary.write \
.mode("overwrite") \
.saveAsTable("gold_network_summary")

StatementMeta(, 1815d0f5-814b-4c4d-8bfc-dc6a5bd3a614, 118, Finished, Available, Finished, False)

In [112]:
spark.catalog.tableExists("gold_network_summary")

StatementMeta(, 1815d0f5-814b-4c4d-8bfc-dc6a5bd3a614, 119, Finished, Available, Finished, False)

True

In [113]:
spark.table("gold_network_summary").show(truncate=False)

StatementMeta(, 1815d0f5-814b-4c4d-8bfc-dc6a5bd3a614, 120, Finished, Available, Finished, False)

+-------------+---------------+-----------------+----------------+-------------+-------------+-------------------+--------------+---------------------+------------------+---------------+----------------+------------------------+
|total_flights|delayed_flights|cancelled_flights|diverted_flights|avg_dep_delay|avg_arr_delay|total_delay_minutes|delay_rate_pct|cancellation_rate_pct|diversion_rate_pct|on_time_flights|on_time_rate_pct|network_disruption_score|
+-------------+---------------+-----------------+----------------+-------------+-------------+-------------------+--------------+---------------------+------------------+---------------+----------------+------------------------+
|547271       |118522         |20389.0          |1512.0          |15.7         |10.35        |8280743.0          |21.66         |3.73                 |0.28              |428749         |78.34           |12.0                    |
+-------------+---------------+-----------------+----------------+-------------+----

In [114]:
spark.catalog.tableExists("gold_network_summary")

StatementMeta(, 1815d0f5-814b-4c4d-8bfc-dc6a5bd3a614, 121, Finished, Available, Finished, False)

True

In [115]:
spark.table("gold_network_summary").show(truncate=False)

StatementMeta(, 1815d0f5-814b-4c4d-8bfc-dc6a5bd3a614, 122, Finished, Available, Finished, False)

+-------------+---------------+-----------------+----------------+-------------+-------------+-------------------+--------------+---------------------+------------------+---------------+----------------+------------------------+
|total_flights|delayed_flights|cancelled_flights|diverted_flights|avg_dep_delay|avg_arr_delay|total_delay_minutes|delay_rate_pct|cancellation_rate_pct|diversion_rate_pct|on_time_flights|on_time_rate_pct|network_disruption_score|
+-------------+---------------+-----------------+----------------+-------------+-------------+-------------------+--------------+---------------------+------------------+---------------+----------------+------------------------+
|547271       |118522         |20389.0          |1512.0          |15.7         |10.35        |8280743.0          |21.66         |3.73                 |0.28              |428749         |78.34           |12.0                    |
+-------------+---------------+-----------------+----------------+-------------+----

In [116]:
tables = [
    "dim_date",
    "dim_airport",
    "dim_airline",
    "dim_route",
    "fact_flights",
    "gold_airport_operations",
    "gold_airline_operations",
    "gold_route_operations",
    "gold_network_summary"
]

for tbl in tables:
    print(
        f"{tbl} :",
        spark.table(tbl).count()
    )

StatementMeta(, 1815d0f5-814b-4c4d-8bfc-dc6a5bd3a614, 123, Finished, Available, Finished, False)

dim_date : 31
dim_airport : 6072
dim_airline : 1536
dim_route : 5666
fact_flights : 547271
gold_airport_operations : 334
gold_airline_operations : 15
gold_route_operations : 5666
gold_network_summary : 1
